#1. DQN
**DQN** - это алгоритм, который объединяет Q-learning с аппроксимацией функции ценности с помощью нейронной сети. В отличие от табличного Q-learning, где Q(s,a) хранится для каждой пары состояние‑действие, DQN использует нейронную сеть с весами θ для аппроксимации Q(s,a; θ). Это позволяет работать с большими и непрерывными пространствами состояний.

Обучение происходит путём минимизации квадратичной ошибки между текущим предсказанием и целевым значением:$$L(\theta) = \mathbb{E}_{(s,a,r,s')} \left[ \left( Q(s,a; \theta) - \left( r + \gamma \max_{a'} Q(s', a'; \theta^-) \right) \right)^2 \right]$$

Важная особенность DQN - использование опыта повторного воспроизведения и целевой сети. Эти механизмы устраняют корреляцию между последовательными переходами и стабилизируют обучение, которое иначе расходилось бы из‑за нестационарности целевых значений. DQN положил начало эре глубокого обучения с подкреплением и показал отличные результаты на многих играх Atari.
#2. Experience raplay
Experience replay - это техника, при которой агент сохраняет свои переходы (s,a,r,s') в буфере (replay buffer) в течение нескольких эпизодов. Затем во время обучения он случайно выбирает мини-батч из этого буфера и использует его для обновления Q-сети.Это нужно для того, чтобы нарушить временную корреляцию между последовательными переходами - в естественной траектории соседние переходы сильно зависимы, что вносит смещение в градиент, а случайная выборка делает данные ближе к независимым и одинаково распределённым. Кроме того, experience replay позволяет многократно использовать редкие или важные переходы, повышая эффективность использования данных, и сглаживает распределение по состояниям и действиям. Experience replay работает только с off-policy алгоритмами (например, Q-learning), потому что они могут обучаться на данных, собранных любой политикой, тогда как on-policy методы (например, SARSA) требуют свежих переходов от текущей политики и поэтому не могут использовать старый опыт.
#3. Target network
Целевая сеть (target network) - это вторая нейронная сеть, имеющая ту же архитектуру, что и основная Q-сеть, но с замороженными весами θ−. Она используется для вычисления целевого значения в функции потерь DQN:
$$y = r + \gamma \max_{a'} Q(s', a'; \theta^-)$$
в базовом Q-learning целевое значение зависит от тех же весов θ
, которые обновляются на каждом шаге. Это приводит к тому, что сеть пытается приблизиться к цели, которая постоянно меняется, что порождает нестабильность и расхождение.

Веса target network обновляются периодическипростым копированием
θ-←θ. Существует также мягкое обновление:
θ−←(1-τ)θ−+τθ с малым τ. Благодаря этому целевые значения меняются медленно, обучение становится устойчивым.
#4. Double DQN
В стандартном DQN используется оператор
maxa'Q(s',a'), который систематически завышает Q-значения из‑за того, что аппроксимационная ошибка распределена симметрично, а максимум берёт положительные выбросы:$$\mathbb{E}[\max_i X_i] \geq \max_i \mathbb{E}[X_i].$$
**Решение (Double DQN)**: разделить выбор действия и его оценку:
$$y = r + \gamma Q(s', \arg \max_{a'} Q(s', a'; \theta); \theta^-)$$
где θ - веса основной сети, θ- - веса целевой сети. То есть действие выбирается основной сетью, а оценивается целевой. Это значительно уменьшает переоценку, поскольку ошибки двух сетей частично некоррелированы и не усиливают друг друга.